In [5]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import ADASYN, SMOTE
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                            balanced_accuracy_score, confusion_matrix,
                            matthews_corrcoef, cohen_kappa_score, log_loss,
                            mean_squared_error, mean_absolute_error, r2_score,
                            roc_curve, auc, roc_auc_score)
import time
import os
import logging

# Set up logging for better tracking
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("classification.log"),
        logging.StreamHandler()
    ]
)

# Constants
K_FOLDS = 5
RANDOM_STATE = 42
SAMPLE_SIZE = 500  # Set sample size for faster processing

# Create directories to save results
os.makedirs("confusion_matrices", exist_ok=True)
os.makedirs("roc_curves", exist_ok=True)
os.makedirs("bar_charts", exist_ok=True)
os.makedirs("class_distributions", exist_ok=True)
os.makedirs("feature_importance", exist_ok=True)
os.makedirs("resampling_evaluation", exist_ok=True)

logging.info("Starting data processing and model training")

# Read the dataset from CSV file
logging.info("Reading dataset from file")
# Update this path to match your file location
df = pd.read_csv("../../../datasets/dataset-2.csv")
logging.info(f"Original dataset shape: {df.shape}")

# Check for missing values and handle them
missing_values = df.isnull().sum().sum()
if missing_values > 0:
    logging.info(f"Found {missing_values} missing values. Filling with mean values.")
    df = df.fillna(df.mean())

# Check for infinite values and replace them
df = df.replace([np.inf, -np.inf], np.nan)
if df.isnull().sum().sum() > 0:
    logging.info("Replacing NaN values with mean...")
    df = df.fillna(df.mean())

# Take sample of the data for faster processing
if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=RANDOM_STATE)
    logging.info(f"Sampled dataset shape: {df.shape}")

# Rename the last column as 'label' if needed
if df.columns[-1] != 'label':
    df.rename(columns={df.columns[-1]: 'label'}, inplace=True)

# Display class distribution
logging.info("Class distribution before resampling:")
class_dist = df['label'].value_counts()
logging.info(class_dist)

# Create visualization of initial class distribution
plt.figure(figsize=(12, 6))
class_dist.plot(kind='bar')
plt.title('Initial Class Distribution')
plt.xlabel('Class')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("resampling_evaluation/initial_class_distribution.png")
plt.close()

X = df.drop(columns=['label']).values
y = df['label'].values
feature_names = df.drop(columns=['label']).columns

# Define resampling techniques
resampling_methods = {
    'None': None,
    'SMOTE': SMOTE(random_state=RANDOM_STATE),
    'ADASYN': ADASYN(random_state=RANDOM_STATE)
}

# Define classifiers with hyperparameters for grid search
classifiers = {
    'DecisionTree': {
        'estimator': DecisionTreeClassifier(random_state=RANDOM_STATE),
        'param_grid': {
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5],
            'criterion': ['gini', 'entropy']
        }
    },
    'KNN': {
        'estimator': KNeighborsClassifier(),
        'param_grid': {
            'n_neighbors': [3, 5, 7],
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan']
        }
    },
    'LogisticRegression': {
        'estimator': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
        'param_grid': {
            'C': [0.1, 1, 10],
            'solver': ['lbfgs', 'liblinear'],
            'penalty': ['l2']
        }
    },
    'RandomForest': {
        'estimator': RandomForestClassifier(random_state=RANDOM_STATE),
        'param_grid': {
            'n_estimators': [50, 100],
            'max_depth': [None, 10],
            'min_samples_split': [2, 5]
        }
    },
    'GradientBoosting': {
        'estimator': GradientBoostingClassifier(random_state=RANDOM_STATE),
        'param_grid': {
            'n_estimators': [50, 100],
            'learning_rate': [0.1],
            'max_depth': [3]
        }
    }
}

# Store results
results = []
timing_results = []
resampling_results = []
best_models = {}

kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=RANDOM_STATE)

# Helper function for confusion matrix metrics
def confusion_matrix_metrics(cm, classes):
    metrics = {}
    for idx, class_label in enumerate(classes):
        TP = cm[idx, idx]  # True Positives for this class
        FP = cm[:, idx].sum() - TP  # False Positives for this class
        FN = cm[idx, :].sum() - TP  # False Negatives for this class
        TN = cm.sum() - (TP + FP + FN)  # True Negatives for this class
 
        metrics[class_label] = {
            'TPR': TP / (TP + FN + 1e-10) if (TP + FN) > 0 else 0,
            'TNR': TN / (TN + FP + 1e-10) if (TN + FP) > 0 else 0,
            'FPR': FP / (FP + TN + 1e-10) if (FP + TN) > 0 else 0,
            'FNR': FN / (FN + TP + 1e-10) if (FN + TP) > 0 else 0
        }
    return metrics

# Function to plot ROC curves
def plot_roc_curve(y_test, y_proba, class_labels, clf_name, fold_idx, resample_name='None'):
    plt.figure(figsize=(10, 8))
    
    # For multiclass, we create one-vs-rest ROC curves
    if len(class_labels) > 2:
        # Initialize arrays to store TPR and FPR for micro and macro averaging
        all_fpr = np.unique(np.concatenate([np.linspace(0, 1, 100) for i in range(len(class_labels))]))
        mean_tpr = np.zeros_like(all_fpr)
        
        for i, class_label in enumerate(class_labels):
            # Convert to binary one-vs-rest classification
            y_test_binary = (y_test == class_label).astype(int)
            
            # Get the probabilities for the current class
            if i < y_proba.shape[1]:
                y_proba_class = y_proba[:, i]
            else:
                # Handle case where there are more classes than probability columns
                y_proba_class = np.zeros(len(y_test))
            
            # Calculate ROC
            fpr, tpr, _ = roc_curve(y_test_binary, y_proba_class)
            
            # Calculate AUC
            roc_auc = auc(fpr, tpr)
            
            # Plot ROC for each class
            plt.plot(fpr, tpr, lw=2,
                     label=f'ROC curve of class {class_label} (area = {roc_auc:.2f})')
            
            # Interpolate mean TPR at specific FPR points (for macro averaging)
            mean_tpr += np.interp(all_fpr, fpr, tpr)
        
        # Compute macro-average ROC curve
        mean_tpr /= len(class_labels)
        macro_roc_auc = auc(all_fpr, mean_tpr)
        plt.plot(all_fpr, mean_tpr, 'b--',
                 label=f'Macro-average ROC (area = {macro_roc_auc:.2f})',
                 lw=3)
    else:
        # Binary classification case
        y_test_binary = y_test
        y_proba_binary = y_proba[:, 1] if y_proba.shape[1] > 1 else y_proba
        fpr, tpr, _ = roc_curve(y_test_binary, y_proba_binary)
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    
    # Add baseline
    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC Curve - {clf_name} with {resample_name} (Fold {fold_idx})')
    plt.legend(loc="lower right")
    plt.savefig(f"roc_curves/{resample_name}_{clf_name}_fold{fold_idx}_roc.png")
    plt.close()
    
    return roc_auc

# Plot class distribution for each resampling method
def plot_class_distribution(y, method_name, fold_idx):
    plt.figure(figsize=(10, 6))
    unique, counts = np.unique(y, return_counts=True)
    plt.bar(unique.astype(str), counts)
    plt.title(f"Class Distribution - {method_name} - Fold {fold_idx}")
    plt.xlabel("Class")
    plt.ylabel("Frequency")
    plt.savefig(f"class_distributions/{method_name}_fold_{fold_idx}.png")
    plt.close()

# Plot feature importance for tree-based models
def plot_feature_importance(clf, feature_names, model_name, resample_name, fold_idx):
    if hasattr(clf, 'feature_importances_'):
        plt.figure(figsize=(12, 8))
        importances = clf.feature_importances_
        indices = np.argsort(importances)[::-1]
        
        # Plot the top 20 features or all if less than 20
        top_n = min(20, len(feature_names))
        plt.barh(range(top_n), importances[indices[:top_n]], align='center')
        plt.yticks(range(top_n), [feature_names[i] for i in indices[:top_n]])
        plt.xlabel('Feature Importance')
        plt.title(f'Top Features - {model_name} with {resample_name} - Fold {fold_idx}')
        plt.tight_layout()
        plt.savefig(f"feature_importance/{model_name}_{resample_name}_fold_{fold_idx}.png")
        plt.close()

logging.info("Starting model training and evaluation...")

# Iterate over resampling methods
for resample_name, resampler in resampling_methods.items():
    logging.info(f"\n===== Processing with resampling method: {resample_name} =====")
    
    # Iterate over classifiers
    for clf_name, clf_info in classifiers.items():
        logging.info(f"Processing classifier: {clf_name}")
        base_estimator = clf_info['estimator']
        param_grid = clf_info['param_grid']
        
        fold_idx = 1
        fold_aucs = []
        
        for train_index, test_index in kf.split(X):
            logging.info(f"Processing fold {fold_idx} for {clf_name} with {resample_name}")
            
            # Split the data
            X_train, X_test = X[train_index], X[test_index]
            y_train, y_test = y[train_index], y[test_index]
            
            # Scale features
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            # Record original class distribution
            original_class_counts = pd.Series(y_train).value_counts().sort_index()
            logging.info(f"Original training set class distribution: {original_class_counts.to_dict()}")
            
            # Apply resampling technique if specified
            if resampler is not None:
                try:
                    # Check if we have enough samples for ADASYN (which requires more samples than SMOTE)
                    min_samples_per_class = original_class_counts.min()
                    
                    if resample_name == 'ADASYN' and min_samples_per_class < 6:
                        logging.info(f"Using SMOTE instead of ADASYN due to insufficient samples (min: {min_samples_per_class})")
                        temp_resampler = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(5, min_samples_per_class-1))
                        X_train_scaled, y_train = temp_resampler.fit_resample(X_train_scaled, y_train)
                        resampling_method_used = "SMOTE"
                    else:
                        X_train_scaled, y_train = resampler.fit_resample(X_train_scaled, y_train)
                        resampling_method_used = resample_name
                    
                    # Log resampling results
                    resampled_class_counts = pd.Series(y_train).value_counts().sort_index()
                    logging.info(f"Resampled training set class distribution: {resampled_class_counts.to_dict()}")
                    
                    # Plot resampled class distribution
                    plot_class_distribution(y_train, f"{resample_name}_{clf_name}", fold_idx)
                    
                    # Save resampling data for analysis
                    resampling_stats = {
                        'Fold': fold_idx,
                        'Classifier': clf_name,
                        'Resampling Method': resampling_method_used,
                        'Original Samples': len(y[train_index]),
                        'Resampled Samples': len(y_train),
                        'Original Classes': len(original_class_counts),
                        'Original Min Samples': min_samples_per_class,
                        'Original Distribution': original_class_counts.to_dict(),
                        'Resampled Distribution': resampled_class_counts.to_dict()
                    }
                    resampling_results.append(resampling_stats)
                    
                except Exception as e:
                    logging.error(f"Resampling failed: {e}. Using original data.")
                    resampling_method_used = "None (Error)"
            else:
                resampling_method_used = "None"
            
            # Record start time for grid search
            full_start_time = time.time()
            
            # Perform grid search for hyperparameter tuning
            grid_search = GridSearchCV(
                estimator=base_estimator,
                param_grid=param_grid,
                cv=3,  # Use 3-fold CV within the training set
                scoring='balanced_accuracy',
                n_jobs=-1
            )
            
            # Train the model with grid search
            start_train_time = time.time()
            grid_search.fit(X_train_scaled, y_train)
            train_time = time.time() - start_train_time
            
            # Get the best model
            best_model = grid_search.best_estimator_
            best_params = grid_search.best_params_
            
            logging.info(f"Best parameters for {clf_name} with {resample_name} on fold {fold_idx}:")
            logging.info(best_params)
            
            # Plot feature importance for tree-based models
            if clf_name in ['DecisionTree', 'RandomForest', 'GradientBoosting']:
                plot_feature_importance(best_model, feature_names, clf_name, resample_name, fold_idx)
            
            # Make predictions
            start_test_time = time.time()
            y_pred = best_model.predict(X_test_scaled)
            
            # Get prediction probabilities for ROC curve
            if hasattr(best_model, "predict_proba"):
                y_proba = best_model.predict_proba(X_test_scaled)
            else:
                # For models without predict_proba, create dummy probabilities
                y_proba = np.zeros((len(y_pred), len(np.unique(y))))
                for i, val in enumerate(y_pred):
                    y_proba[i, val] = 1
            
            test_time = time.time() - start_test_time
            
            # Record total time including grid search
            full_time = time.time() - full_start_time
            
            timing_results.append({
                'Resampling': resampling_method_used,
                'Classifier': clf_name,
                'Fold': fold_idx,
                'Grid Search Time (s)': full_time - train_time - test_time,
                'Training Time (s)': train_time,
                'Testing Time (s)': test_time,
                'Total Time (s)': full_time
            })
            
            # Store best model for this combination
            model_key = f"{resample_name}_{clf_name}_fold_{fold_idx}"
            best_models[model_key] = {
                'model': best_model,
                'params': best_params,
                'balanced_accuracy': balanced_accuracy_score(y_test, y_pred)
            }
            
            # Compute metrics
            unique_classes = np.unique(np.concatenate([y_test, y_pred]))
            cm = confusion_matrix(y_test, y_pred, labels=unique_classes)
            cm_metrics = confusion_matrix_metrics(cm, unique_classes)
            
            # Plot ROC curve
            try:
                roc_auc = plot_roc_curve(y_test, y_proba, unique_classes, clf_name, fold_idx, resample_name)
                fold_aucs.append(roc_auc)
            except Exception as e:
                logging.error(f"Error plotting ROC curve: {e}")
            
            # Calculate overall metrics
            overall_metrics = {
                'Resampling': resampling_method_used,
                'Classifier': clf_name,
                'Fold': fold_idx,
                'Class': 'Overall',
                'Accuracy': accuracy_score(y_test, y_pred),
                'Precision': precision_score(y_test, y_pred, average='weighted', zero_division=0),
                'Recall': recall_score(y_test, y_pred, average='weighted', zero_division=0),
                'F1 Score': f1_score(y_test, y_pred, average='weighted', zero_division=0),
                'Balanced Accuracy': balanced_accuracy_score(y_test, y_pred),
                'Matthews Correlation': matthews_corrcoef(y_test, y_pred),
                'Cohen Kappa': cohen_kappa_score(y_test, y_pred),
                'Best Parameters': str(best_params),
                'Training Time (s)': train_time,
                'Testing Time (s)': test_time
            }
            results.append(overall_metrics)
            
            # Calculate per-class metrics
            class_metrics_list = []
            for class_label in unique_classes:
                # One-vs-Rest approach for metrics
                y_test_binary = (y_test == class_label).astype(int)
                y_pred_binary = (y_pred == class_label).astype(int)
                
                class_specific_metrics = {
                    'Resampling': resampling_method_used,
                    'Classifier': clf_name,
                    'Fold': fold_idx,
                    'Class': class_label,
                    'Precision': precision_score(y_test_binary, y_pred_binary, zero_division=0),
                    'Recall': recall_score(y_test_binary, y_pred_binary, zero_division=0),
                    'F1 Score': f1_score(y_test_binary, y_pred_binary, zero_division=0),
                    'Balanced Accuracy': balanced_accuracy_score(y_test_binary, y_pred_binary),
                    'True Positive Rate (TPR)': cm_metrics[class_label]['TPR'],
                    'True Negative Rate (TNR)': cm_metrics[class_label]['TNR'],
                    'False Positive Rate (FPR)': cm_metrics[class_label]['FPR'],
                    'False Negative Rate (FNR)': cm_metrics[class_label]['FNR'],
                }
                class_metrics_list.append(class_specific_metrics)
            
            # Plot and save confusion matrix
            plt.figure(figsize=(10, 8))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                        xticklabels=unique_classes, yticklabels=unique_classes)
            plt.title(f"{clf_name} with {resampling_method_used} - Fold {fold_idx} Confusion Matrix")
            plt.xlabel("Predicted")
            plt.ylabel("True")
            plt.tight_layout()
            plt.savefig(f"confusion_matrices/{resampling_method_used}_{clf_name}_fold{fold_idx}_cm.png")
            plt.close()
            
            # Append results for this fold
            results.extend(class_metrics_list)
            fold_idx += 1

# Save resampling results
resampling_df = pd.DataFrame(resampling_results)
resampling_df.to_csv("resampling_evaluation/resampling_results.csv", index=False)
logging.info("\nResampling results saved to resampling_evaluation/resampling_results.csv")

# Save timing results
timing_df = pd.DataFrame(timing_results)
timing_df.to_csv("timing_results.csv", index=False)
logging.info("\nTiming results saved to timing_results.csv")

# Create a DataFrame for results
results_df = pd.DataFrame(results)
logging.info("\nClassification Metrics Across Resampling Methods and Folds:")

# Fix: Filter to only numerical columns for aggregation and select only 'Overall' rows
# This is the fix for the TypeError issue
metrics_columns = ['Accuracy', 'F1 Score', 'Balanced Accuracy']
agg_df = results_df[results_df['Class'] == 'Overall'].groupby(['Resampling', 'Classifier'])[metrics_columns].mean()
logging.info(agg_df)

# Save results to CSV
results_df.to_csv("metrics_results.csv", index=False)
logging.info("\nMetrics results saved to metrics_results.csv")

# Find the best model across all combinations
overall_metrics = results_df[results_df['Class'] == 'Overall']
best_combination = overall_metrics.loc[overall_metrics['Balanced Accuracy'].idxmax()]
logging.info("\nBest Model Combination:")
logging.info(f"Resampling: {best_combination['Resampling']}")
logging.info(f"Classifier: {best_combination['Classifier']}")
logging.info(f"Balanced Accuracy: {best_combination['Balanced Accuracy']:.4f}")
logging.info(f"F1 Score: {best_combination['F1 Score']:.4f}")
logging.info(f"Parameters: {best_combination['Best Parameters']}")

# Generate summary statistics - Fix for the error
# Only include numeric columns in aggregation
numeric_cols = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'Balanced Accuracy', 
                'Matthews Correlation', 'Cohen Kappa', 'Training Time (s)', 'Testing Time (s)']

# Create separate dataframes for mean and std to avoid the error
summary_mean = results_df[results_df['Class'] == 'Overall'].groupby(['Resampling', 'Classifier'])[numeric_cols].mean()
summary_std = results_df[results_df['Class'] == 'Overall'].groupby(['Resampling', 'Classifier'])[numeric_cols].std()

# Create a MultiIndex columns dataframe for combined statistics
columns_multi = pd.MultiIndex.from_product([numeric_cols, ['mean', 'std']])
summary_df = pd.DataFrame(index=summary_mean.index, columns=columns_multi)

# Fill the MultiIndex dataframe
for col in numeric_cols:
    summary_df[(col, 'mean')] = summary_mean[col]
    summary_df[(col, 'std')] = summary_std[col]

summary_df.to_csv("summary_metrics.csv")
logging.info("\nSummary metrics saved to summary_metrics.csv")

# Create comparison plots between resampling methods
plt.figure(figsize=(15, 10))
sns.barplot(x='Classifier', y='F1 Score', hue='Resampling', 
            data=results_df[results_df['Class'] == 'Overall'])
plt.title('F1 Score Comparison Across Resampling Methods and Classifiers')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('bar_charts/f1_comparison.png')
plt.close()

plt.figure(figsize=(15, 10))
sns.barplot(x='Classifier', y='Balanced Accuracy', hue='Resampling', 
            data=results_df[results_df['Class'] == 'Overall'])
plt.title('Balanced Accuracy Comparison Across Resampling Methods and Classifiers')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('bar_charts/balanced_accuracy_comparison.png')
plt.close()

# Plot timing information
plt.figure(figsize=(15, 10))
sns.barplot(x='Classifier', y='Total Time (s)', hue='Resampling', data=timing_df)
plt.title('Total Processing Time Comparison')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('bar_charts/time_comparison.png')
plt.close()

logging.info("\nAnalysis complete! Check CSV files and visualizations for detailed results.")

2025-04-08 10:56:33,105 - INFO - Starting data processing and model training
2025-04-08 10:56:33,106 - INFO - Reading dataset from file
2025-04-08 10:56:33,625 - INFO - Original dataset shape: (218805, 47)
2025-04-08 10:56:33,773 - INFO - Sampled dataset shape: (500, 47)
2025-04-08 10:56:33,774 - INFO - Class distribution before resampling:
2025-04-08 10:56:33,775 - INFO - label
6     74
14    60
10    50
12    49
13    44
8     44
21    37
9     37
20    30
19    18
1     15
23     8
25     6
4      6
24     6
7      5
27     4
16     2
15     2
31     1
29     1
22     1
Name: count, dtype: int64
2025-04-08 10:56:33,964 - INFO - Starting model training and evaluation...
2025-04-08 10:56:33,965 - INFO - 
===== Processing with resampling method: None =====
2025-04-08 10:56:33,966 - INFO - Processing classifier: DecisionTree
2025-04-08 10:56:33,967 - INFO - Processing fold 1 for DecisionTree with None
2025-04-08 10:56:33,970 - INFO - Original training set class distribution: {1: 13, 4: 